# Databricks Validation Notebook

This notebook validates the environment setup for the Spark Sensei project.

It installs required libraries from requirements.txt, sets up a secure API key input using dbutils.widgets, and runs a validation script to confirm that all libraries are installed and the Groq API is reachable.

In [ ]:
%pip install --upgrade typing-extensions
%pip install -r requirements.txt

In [ ]:
# Secure API Key Input
dbutils.widgets.text("api_key", "", "API Key")

In [ ]:
# Validate Library Imports
try:
    import langchain
    import langgraph
    import mlflow
    from langchain_groq import ChatGroq
    print("✓ All required libraries imported successfully!")
    print(f"  - langchain: {langchain.__version__}")
    print(f"  - langgraph: {langgraph.__version__}")
    print(f"  - mlflow: {mlflow.__version__}")
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("💡 Try restarting the cluster or upgrading typing-extensions:")
    print("   %pip install --upgrade typing-extensions")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

In [ ]:
# Check typing_extensions version
import typing_extensions
print(f"typing_extensions version: {typing_extensions.__version__}")

# Test Sentinel import
try:
    from typing_extensions import Sentinel
    print("✓ Sentinel import successful")
except ImportError as e:
    print(f"❌ Sentinel import failed: {e}")
    print("🔧 Installing compatible version...")
    # This will be handled by the pip install above

In [ ]:
# Databricks Secrets Validation
try:
    # Check if Databricks secrets are accessible
    host_secret = dbutils.secrets.get(scope="your-secret-scope", key="databricks-host")
    token_secret = dbutils.secrets.get(scope="your-secret-scope", key="databricks-token")

    if not host_secret:
        print("ERROR: DATABRICKS_HOST_SECRET is not set")
        raise ValueError("Missing Databricks host secret")

    if not token_secret:
        print("ERROR: DATABRICKS_TOKEN is not set")
        raise ValueError("Missing Databricks token secret")

    print("✓ Databricks secrets are available")

except Exception as e:
    print(f"Secret validation failed: {e}")
    print("Note: Make sure you have created a secret scope and stored your Databricks host and token.")
    print("You can create secrets in the Databricks UI under 'User Settings' > 'Developer' > 'Access tokens'")
    print("Or use dbutils.secrets.createScope() to create a scope programmatically")

In [ ]:
# Validate API Reachability
from langchain_groq import ChatGroq

api_key = dbutils.widgets.get("api_key")
if not api_key:
    print("API key not provided.")
else:
    try:
        llm = ChatGroq(api_key=api_key)
        response = llm.invoke("Hello, this is a test message.")
        print("API is reachable! Response received.")
    except Exception as e:
        print(f"API reachability test failed: {e}")

In [ ]:
# Load samples.nyctaxi.trips into temporary view
spark.sql("CREATE OR REPLACE TEMP VIEW trips AS SELECT * FROM samples.nyctaxi.trips")

# Verify by selecting first 5 rows
result = spark.sql("SELECT * FROM trips LIMIT 5")
result.show()